In [1]:
# This will be used to upload golf ranking data to db 
# This first version is the inital upload; will need to be tweaked for regular refresh 

In [2]:
import os
import pandas as pd
import re
import numpy as np
from datetime import datetime

In [3]:
# Need to re-do postgres creds 
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values

# Load environment variables from .env file
load_dotenv()

# Database configuration from environment variables
db_config = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': os.getenv('DB_PORT', '5432')
}
schema = os.getenv('DB_SCHEMA')

# Test the connection
try:
    conn = psycopg2.connect(**db_config)
    conn.close()
    print("✅ Database connection successful!")
    print(f"   Connected to: {db_config['database']} on {db_config['host']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
   Connected to: brdb on localhost


In [4]:
# Folder where your CSV files are stored
# csv_folder = '/Users/brandon/Desktop/Golf_Data/datagolf_rankings_current.csv' # mac version 
csv_folder = r"C:\Users\brand\OneDrive\Documents\pga_stats\upload_staging\data_golf\datagolf_rankings_current.csv"

In [5]:
# use this to insert one file into a table for sure 
# basically create a table and then load up one test file 
'''
import psycopg2
import pandas as pd
import os
from datetime import datetime


conn = psycopg2.connect(
    dbname='brdb',
    user='brandon',
    password='',
    host='localhost',
    port='5432'
)

cur = conn.cursor()
'''


"\nimport psycopg2\nimport pandas as pd\nimport os\nfrom datetime import datetime\n\n\nconn = psycopg2.connect(\n    dbname='brdb',\n    user='brandon',\n    password='',\n    host='localhost',\n    port='5432'\n)\n\ncur = conn.cursor()\n"

In [6]:

create_table_query = """
CREATE TABLE IF NOT EXISTS datagolf_ranks (
    id SERIAL PRIMARY KEY,
    player_name VARCHAR(100), 
    primary_tour VARCHAR(100),
    dg_rank INT,
    owgr_rank INT,
    dg_index NUMERIC(10,3),
    refresh_date TIMESTAMP
     
);
"""
conn = psycopg2.connect(**db_config)
cur = conn.cursor()

cur.execute(create_table_query)
conn.commit()


In [7]:
# Commit changes
conn.commit()
print("Table created successfully!")

Table created successfully!


In [8]:
# Read CSV file into a pandas DataFrame
df = pd.read_csv(csv_folder, index_col=False)

# Add refresh date column
df['refresh_date'] = datetime.now()

# Drop unwanted columns
columns_to_drop = ['dg_change', 'owgr_change','dg_points_rank', 'dg_points_change']
df = df.drop(columns=columns_to_drop, errors='ignore')

df['owgr_rank'] = pd.to_numeric(df['owgr_rank'], errors='coerce')
df['dg_rank'] = pd.to_numeric(df['dg_rank'], errors='coerce')



df = df.where(pd.notnull(df), None)
df['owgr_rank'] = df['owgr_rank'].fillna(9999)
numeric_columns = ['owgr_rank', 'dg_rank', 'dg_index']
for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].where(pd.notna(df[col]), None)


print(df)

            player_name     primary_tour  dg_rank  dg_index  owgr_rank  \
0    Scheffler, Scottie         PGA Tour        1     3.055        1.0   
1             Rahm, Jon         LIV Golf        2     1.951       54.0   
2         McIlroy, Rory    DP World Tour        3     1.941        2.0   
3      Fleetwood, Tommy         PGA Tour        4     1.910        3.0   
4    Schauffele, Xander         PGA Tour        5     1.607       10.0   
..                  ...              ...      ...       ...        ...   
495      Doyle, Brendon      Other Tours      496    -1.170     9999.0   
496     Kinhult, Marcus    DP World Tour      497    -1.171      422.0   
497      Zheng, Sampson      Other Tours      498    -1.176     9999.0   
498          Hall, Ryan  Korn Ferry Tour      499    -1.176     9999.0   
499         Yuan, Kevin      Other Tours      500    -1.179     9999.0   

                  refresh_date  
0   2026-03-03 16:19:11.507533  
1   2026-03-03 16:19:11.507533  
2   2026-03-

In [9]:
# ...existing code...

# Convert "Last, First" to "First Last"
df['player_name'] = df['player_name'].apply(
    lambda x: ' '.join([part.strip() for part in x.split(',')[::-1]]) if ',' in x else x
)

# ...existing code...

In [10]:
df

,player_name,primary_tour,dg_rank,dg_index,owgr_rank,refresh_date
0,Scottie Scheffler,PGA Tour,1,3.055,1.0,2026-03-03 16:19:11.507533
1,Jon Rahm,LIV Golf,2,1.951,54.0,2026-03-03 16:19:11.507533
2,Rory McIlroy,DP World Tour,3,1.941,2.0,2026-03-03 16:19:11.507533
3,Tommy Fleetwood,PGA Tour,4,1.910,3.0,2026-03-03 16:19:11.507533
4,Xander Schauffele,PGA Tour,5,1.607,10.0,2026-03-03 16:19:11.507533
...,...,...,...,...,...,...
495,Brendon Doyle,Other Tours,496,-1.170,9999.0,2026-03-03 16:19:11.507533
496,Marcus Kinhult,DP World Tour,497,-1.171,422.0,2026-03-03 16:19:11.507533
497,Sampson Zheng,Other Tours,498,-1.176,9999.0,2026-03-03 16:19:11.507533
498,Ryan Hall,Korn Ferry Tour,499,-1.176,9999.0,2026-03-03 16:19:11.507533


In [11]:
# table the data is being inserted into 
table_name = 'datagolf_ranks'

In [12]:
for _, row in df.iterrows():
    columns = ', '.join(f'"{col}"' for col in df.columns)  # quoting for safety
    placeholders = ', '.join(['%s'] * len(row))
    insert_sql = f"INSERT INTO {schema}.{table_name} ({columns}) VALUES ({placeholders})"
    print(row)  # Debug: See the values being inserted
    cur.execute(insert_sql, tuple(row))

conn.commit()
cur.close()
conn.close()
print("Data inserted successfully!")

player_name              Scottie Scheffler
primary_tour                      PGA Tour
dg_rank                                  1
dg_index                             3.055
owgr_rank                              1.0
refresh_date    2026-03-03 16:19:11.507533
Name: 0, dtype: object
player_name                       Jon Rahm
primary_tour                      LIV Golf
dg_rank                                  2
dg_index                             1.951
owgr_rank                             54.0
refresh_date    2026-03-03 16:19:11.507533
Name: 1, dtype: object
player_name                   Rory McIlroy
primary_tour                 DP World Tour
dg_rank                                  3
dg_index                             1.941
owgr_rank                              2.0
refresh_date    2026-03-03 16:19:11.507533
Name: 2, dtype: object
player_name                Tommy Fleetwood
primary_tour                      PGA Tour
dg_rank                                  4
dg_index                    

In [13]:
conn.close()